In [ ]:
!pip install sentence-transformers faiss-cpu google-generativeai torch torch-geometric -q


In [ ]:
# Reddit: copy Kaggle .npz into working dir, then load (else PyG Reddit download).
import glob
import logging
import os
import shutil
import numpy as np
import torch
from torch_geometric.datasets import Reddit
from torch_geometric.data import Data

REDDIT_ROOT = os.environ.get("REDDIT_DATA_ROOT", "/kaggle/working/reddit_data")
REDDIT_NPZ_SOURCE = os.environ.get(
    "REDDIT_NPZ_SOURCE", "/kaggle/input/datasets/akshatsharma1011/reddit-files"
)
NPZ_DIR = os.path.join(REDDIT_ROOT, "npz_kaggle")
# Cap graph size for RAG (speed + memory); override with REDDIT_MAX_NODES
REDDIT_MAX_NODES = int(os.environ.get("REDDIT_MAX_NODES", "250000"))
logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)


def prepare_reddit_npz() -> int:
    if not os.path.isdir(REDDIT_NPZ_SOURCE):
        log.info("Kaggle reddit-files not found: %s", REDDIT_NPZ_SOURCE)
        return 0
    os.makedirs(NPZ_DIR, exist_ok=True)
    n = 0
    for p in glob.glob(os.path.join(REDDIT_NPZ_SOURCE, "*.npz")):
        if os.path.isfile(p):
            shutil.copy2(p, os.path.join(NPZ_DIR, os.path.basename(p)))
            n += 1
    log.info("Copied %d .npz -> %s", n, NPZ_DIR)
    return n


def _pick(arrs, *names, required=True):
    for n in names:
        if n in arrs and arrs[n] is not None:
            return arrs[n]
    if required:
        raise KeyError(f"None of {names} in npz; keys: {list(arrs.keys())}")
    return None


def _to_dense_features(feat, num_nodes_hint=None):
    """Dense (N, F) from ndarray or scipy sparse; unbox 0-d object arrays."""
    if feat is None:
        return None
    if isinstance(feat, np.ndarray) and feat.dtype == object and feat.size == 1:
        feat = feat.item()
    try:
        from scipy import sparse as sp
        if sp.issparse(feat):
            feat = feat.toarray()
    except Exception:
        pass
    a = np.asarray(feat, dtype=np.float32)
    if a.ndim == 1 and num_nodes_hint and num_nodes_hint > 0:
        if a.size == num_nodes_hint:
            a = a.reshape(-1, 1)
        elif a.size % num_nodes_hint == 0 and a.size != num_nodes_hint:
            a = a.reshape(num_nodes_hint, -1)
    return a


def _build_edge_index(merged: dict) -> np.ndarray:
    ei = _pick(merged, "edge_index", "edges", required=False)
    if ei is not None:
        ei = np.asarray(ei, dtype=np.int64)
        if ei.shape[0] != 2 and ei.size and ei.shape[1] == 2:
            ei = ei.T
        return ei
    if "row" in merged and "col" in merged:
        r = np.asarray(merged["row"], dtype=np.int64).ravel()
        c = np.asarray(merged["col"], dtype=np.int64).ravel()
        if r.size != c.size:
            raise ValueError(f"row {r.size} vs col {c.size}")
        return np.stack([r, c], axis=0)
    raise KeyError(f"No edge_index/edges or row+col; keys: {list(merged.keys())}")


def load_reddit_from_npz():
    gfile = os.path.join(NPZ_DIR, "reddit_graph.npz")
    dfile = os.path.join(NPZ_DIR, "reddit_data.npz")
    if not (os.path.isfile(gfile) and os.path.isfile(dfile)):
        return None
    merged = {}
    for path in (gfile, dfile):
        z = np.load(path, allow_pickle=True)
        for k in z.files:
            merged[k] = z[k]
    y = _pick(merged, "y", "Y", "labels", "label", "y_train", "y_all")
    y = np.asarray(y).reshape(-1).astype(np.int64)
    num_nodes = int(y.shape[0])
    x = _pick(
        merged, "x", "X", "features", "feat", "feature", "Feature", "node_features", required=False
    )
    if x is None:
        raise KeyError(f"No node features; keys: {list(merged.keys())}")
    x = _to_dense_features(x, num_nodes)
    if x.shape[0] != num_nodes:
        if x.ndim == 2 and x.shape[1] == num_nodes:
            x = x.T
        else:
            raise ValueError(f"feature rows {x.shape[0]} != num_nodes {num_nodes} from label")
    ei = _build_edge_index(merged)
    ei = np.asarray(ei, dtype=np.int64)
    if ei.shape[0] != 2:
        ei = ei.T if ei.shape[1] == 2 else ei
    return Data(
        x=torch.from_numpy(x),
        y=torch.from_numpy(y),
        edge_index=torch.from_numpy(ei).long().contiguous(),
        num_nodes=num_nodes,
    )


def normalize_features(x):
    norm = x.norm(p=1, dim=1, keepdim=True).clamp(min=1.0)
    return x / norm


def subsample_graph(data: Data, max_nodes: int) -> Data:
    """Random subset of nodes; vectorized edge filter (avoids slow Python over edges)."""
    n = int(data.num_nodes)
    if n <= max_nodes:
        return data
    dev = data.x.device
    perm = torch.randperm(n, device=dev)[:max_nodes]
    inv = torch.full((n,), -1, dtype=torch.long, device=dev)
    inv[perm] = torch.arange(max_nodes, device=dev, dtype=torch.long)
    e = data.edge_index.to(dev)
    src, dst = e[0], e[1]
    ok = (inv[src] >= 0) & (inv[dst] >= 0)
    new_ei = torch.stack([inv[src[ok]], inv[dst[ok]]], dim=0).contiguous()
    return Data(
        x=data.x[perm],
        y=data.y[perm],
        edge_index=new_ei,
        num_nodes=max_nodes,
    )


prepare_reddit_npz()
raw = load_reddit_from_npz()
if raw is not None:
    log.info("Using Reddit from npz.")
else:
    log.info("Falling back to PyG Reddit.")
    raw = Reddit(root=REDDIT_ROOT)[0]
if raw.num_nodes > REDDIT_MAX_NODES:
    log.info("Subsample %d -> %d nodes (REDDIT_MAX_NODES)", raw.num_nodes, REDDIT_MAX_NODES)
    raw = subsample_graph(raw, REDDIT_MAX_NODES)
raw.x = normalize_features(raw.x)
oc = int(raw.y.max().item()) + 1
raw.y = (raw.y * 4 // oc).clamp(0, 3)
nodes = list(range(raw.num_nodes))
labels_dict = {i: f"cluster_{int(raw.y[i].item())}" for i in nodes}
print("Total nodes:", len(nodes))


In [ ]:
documents = []
for node in nodes:
    label = labels_dict[node]
    doc = (
        f"Node ID: {node}\n"
        f"This Reddit post node is assigned to community label: {label} (4-way aggregation).\n"
        f"It is part of a post-to-post co-comment network.\n"
    )
    documents.append(doc)
print(documents[:2])


In [ ]:
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = encoder.encode(documents, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)


In [ ]:
import faiss
import numpy as np
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))
print("FAISS index ready")


In [ ]:
import os
import google.generativeai as genai
api_key = os.environ.get("GEMINI_API_KEY", "")
if not api_key:
    raise ValueError("Set GEMINI_API_KEY in the environment (do not hardcode secrets).")
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini ready")


In [ ]:
def retrieve(query, top_k=5):
    q_emb = encoder.encode([query])
    distances, indices = index.search(np.array(q_emb).astype("float32"), top_k)
    return [documents[i] for i in indices[0]]


In [ ]:
def rag_query(query, top_k=5):
    context_docs = retrieve(query, top_k)
    context = "\n".join(context_docs)
    prompt = f"""
    You are a graph intelligence assistant.

    Context from Reddit co-comment graph (post nodes):
    {context}

    Question:
    {query}

    Answer clearly using the context.
    """
    response = model.generate_content(prompt)
    print("Query:", query)
    print("\nRetrieved context:\n")
    for doc in context_docs:
        print("-", doc.strip())
    print("\nAnswer:\n")
    print(response.text)


In [ ]:
rag_query("What kinds of community labels appear in this Reddit graph?")


In [ ]:
rag_query("Summarize the network context.")


In [ ]:
rag_query("Which node IDs are mentioned for cluster_0?")
